In [1]:
#Import Library
import random

In [2]:
#Create Functions
def extended_gcd(a, b):
    if a == 0:
        return b, 0, 1
    else:
        g, x, y = extended_gcd(b % a, a)
        return g, y - (b // a) * x, x

def modular_inverse(a, m):
    g, x, y = extended_gcd(a, m)
    if g != 1:
        raise ValueError("The modular inverse does not exist (a and m are not coprime).")
    else:
        return x % m

In [3]:
#Create Power and Prime Functions
def power(a, b, m):
    """Calculates (a^b) % m using modular exponentiation."""
    res = 1
    a %= m
    while b > 0:
        if b % 2 == 1:
            res = (res * a) % m
        a = (a * a) % m
        b //= 2
    return res

def is_prime(n, k=5):
    """Miller-Rabin primality test to check if n is likely prime.
    k is the number of iterations (higher k means higher certainty).
    """
    if n < 2:
        return False
    if n == 2 or n == 3:
        return True
    if n % 2 == 0:
        return False

    # Write n-1 as 2^s * d
    s = 0
    d = n - 1
    while d % 2 == 0:
        d //= 2
        s += 1

    # Witness loop
    for _ in range(k):
        a = random.randint(2, n - 2)
        x = power(a, d, n)
        if x == 1 or x == n - 1:
            continue
        composite = True
        for _ in range(s - 1):
            x = power(x, 2, n)
            if x == n - 1:
                composite = False
                break
        if composite:
            return False # n is composite
    return True # n is probably prime

In [4]:
#Function to Create a a random number of 'bits' length
def generate_prime(bits):
    while True:
        num = random.getrandbits(bits)
        num |= (1 << bits - 1)
        num |= 1

        if is_prime(num):
            if num % 4 == 3:
                return num

In [5]:
#Function to Generate Keys
def generate_keys(bits):
    while True:
        p = generate_prime(bits) # Generate first prime p
        q = generate_prime(bits) # Generate second prime q
        if p != q:
            break

    n = p * q
    b = 0

    public_key = (n, b)
    private_key = (p, q)

    return public_key, private_key

In [6]:
#Encryption Function
def encrypt(m, public_key):
    n, b = public_key

    # Check if the message is within the valid range [0, n-1]
    if not (0 <= m < n):
        raise ValueError("Plaintext message m must be in the range [0, n-1].")

    # Encryption formula: c = m * (m + b) mod n
    c = (m * (m + b)) % n
    return c

In [7]:
#Find the Square Roots in Prime Values - Function
def find_square_roots_mod_prime(c, p):
    exponent = (p + 1) // 4
    r1 = power(c, exponent, p)
    r2 = p - r1
    return r1, r2

In [8]:
#Function to Apply Chinese Remainder Theorem
def chinese_remainder_theorem(a1, n1, a2, n2):
    g, x, y = extended_gcd(n1, n2)
    if g != 1:
        raise ValueError("n1 and n2 are not coprime, CRT not applicable directly.")

    inv_n1_mod_n2 = modular_inverse(n1, n2)
    inv_n2_mod_n1 = modular_inverse(n2, n1)

    result = (a1 * n2 * inv_n2_mod_n1 + a2 * n1 * inv_n1_mod_n2) % (n1 * n2)
    return result

In [9]:
#Decrypt Function
def decrypt(c, private_key, b):
    if b != 0:
        raise NotImplementedError("Decryption for b != 0 is more complex and not implemented here.")

    p, q = private_key
    n = p * q

    r_p1, r_p2 = find_square_roots_mod_prime(c, p)
    r_q1, r_q2 = find_square_roots_mod_prime(c, q)
    plaintexts = []

    plaintexts.append(chinese_remainder_theorem(r_p1, p, r_q1, q))
    plaintexts.append(chinese_remainder_theorem(r_p1, p, r_q2, q))
    plaintexts.append(chinese_remainder_theorem(r_p2, p, r_q1, q))
    plaintexts.append(chinese_remainder_theorem(r_p2, p, r_q2, q))

    return plaintexts

In [10]:
#Application Example
#Step 01 - Key Generation

BIT_LENGTH = 16 # Use a small bit length for demonstration purposes to keep numbers manageable
print(f"\nGenerating Rabin keys with bit length: {BIT_LENGTH}")
public_key, private_key = generate_keys(BIT_LENGTH)
n, b = public_key
p, q = private_key

print(f"Public Key (n, b): ({n}, {b})")
print(f"Private Key (p, q): ({p}, {q})")
print(f"p %% 4: {p % 4}")
print(f"q %% 4: {q % 4}")


Generating Rabin keys with bit length: 16
Public Key (n, b): (2290038209, 0)
Private Key (p, q): (61331, 37339)
p %% 4: 3
q %% 4: 3


In [11]:
#Step 02 - Message Preparation ( Validation )
original_message = 12345

print(f"\nOriginal Message (m): {original_message}")
if not (0 <= original_message < n):
    print(f"Warning: Original message {original_message} is not in range [0, {n-1}]. Adjusting...")

    if original_message >= n:
        original_message = original_message % n
        if original_message == 0:
            original_message = 1
    print(f"Adjusted Message (m): {original_message}")


Original Message (m): 12345


In [12]:
#Step 3 - Encryption
ciphertext = encrypt(original_message, public_key)
print(f"Ciphertext (c): {ciphertext}")

Ciphertext (c): 152399025


In [13]:
#Step 4 - Decryption
decrypted_plaintexts = decrypt(ciphertext, private_key, b)
print(f"Four possible decrypted plaintexts: {decrypted_plaintexts}")

Four possible decrypted plaintexts: [2290025864, 1513698066, 776340143, 12345]


In [14]:
#Step 5 - Plaintext Recovery
recovered_message = None
for m_candidate in decrypted_plaintexts:
    if m_candidate == original_message:
        recovered_message = m_candidate
        break

    if b == 0 and (n - m_candidate) == original_message:
        recovered_message = n - m_candidate
        break

if recovered_message is not None:
    print(f"Original message successfully recovered: {recovered_message}")
else:
    print("Could not unambiguously recover the original message from the four candidates.")
    print(f"Original: {original_message}")
    print(f"Candidates: {decrypted_plaintexts}")

Original message successfully recovered: 12345
